# Обзор техник прунинга нейронных сетей

Прунинг (pruning) — это набор методов сжатия нейронных сетей, основанных на удалении «лишних» параметров: отдельных весов, нейронов, каналов или целых слоёв. Цель — уменьшить размер модели, ускорить инференс и снизить энергопотребление при минимальной потере качества. Идея восходит к работам конца 80-х (LeCun et al., *Optimal Brain Damage*), но настоящий ренессанс произошёл с появлением больших моделей и Lottery Ticket Hypothesis (Frankle & Carbin, 2019).

## Базовая мотивация

Большие нейронные сети сильно перепараметризованы. Эмпирически показано, что значительную часть весов можно обнулить без существенной потери точности. Формально задача прунинга формулируется как поиск маски $m \in \{0, 1\}^{|\theta|}$ такой, что:

$$
\min_{\theta, m} \mathcal{L}(f(x; \theta \odot m)) \quad \text{s.t.} \quad \|m\|_0 \leq k
$$

где $\theta$ — параметры модели, $\odot$ — поэлементное умножение, $k$ — целевое число ненулевых параметров. Поскольку $\ell_0$-ограничение делает задачу NP-трудной, на практике используются эвристики и релаксации.

## Классификация методов

Прунинг можно классифицировать по нескольким независимым осям.

### По структуре

- Неструктурированный (unstructured) прунинг — обнуляются отдельные веса в матрицах. Даёт максимальную гибкость и наилучшие результаты по точности, но требует специального аппаратного и программного обеспечения для реального ускорения (sparse kernels). На обычных GPU выигрыш в скорости почти отсутствует.
- Структурированный (structured) прунинг — удаляются целые структурные единицы: нейроны, каналы свёрток, attention-головы, слои. Сразу даёт ускорение на стандартном железе, но обычно сильнее бьёт по качеству при той же степени разреженности.
- Полуструктурированный (semi-structured) — компромисс. Самый известный вариант — N:M sparsity (например, 2:4 от NVIDIA Ampere): из каждых M подряд идущих весов ровно N должны быть ненулевыми. Поддерживается аппаратно в Tensor Cores.

<img src="img/pruning_strategies.png" width=500>

### По моменту применения

- После обучения (post-training pruning) — прунинг уже обученной модели, обычно с последующим её дообучением с выключенными весами fine-tuning.
- Во время обучения (pruning during training) — маска обновляется параллельно с весами; примеры: gradual magnitude pruning, $\ell_0$-регуляризация.
- До обучения (pruning at initialization) — маска вычисляется на инициализированной сети, например в SNIP, GraSP, SynFlow. Подход интересен теоретически, но на практике уступает методам с обучением.

### По итеративности

- One-shot — обнулить $k\%$ весов за один проход, опционально дообучить.
- Iterative — чередование шагов прунинга и дообучения. Обычно даёт значительно лучшие результаты, особенно при высоких степенях разреженности (>90%).

## Ключевые методы

### Magnitude pruning

Самый простой и удивительно сильный baseline. Обнуляются веса с наименьшим модулем:

$$
m_i = \mathbb{1}[|\theta_i| > \tau]
$$

Порог $\tau$ выбирается глобально или послойно. Метод опирается на интуицию: маленькие веса слабо влияют на выход. Несмотря на простоту, magnitude pruning остаётся конкурентоспособным практически во всех бенчмарках. Итеративная версия (Han et al., 2015; *Deep Compression*) долго была стандартом де-факто.

### Методы на основе кривизны (Hessian-based)

Обнуление веса меняет лосс. Если разложить изменение в ряд Тейлора около локального минимума, линейный член обнуляется и остаётся:

$$
\Delta \mathcal{L} \approx \tfrac{1}{2} \Delta\theta^\top H \Delta\theta
$$

где $H$ — гессиан. Отсюда получают «важность» веса (saliency)

Работы:
- Optimal Brain Damage (LeCun, 1989) использует диагональное приближение гессиана.
- Optimal Brain Surgeon (Hassibi & Stork, 1993) использует полный гессиан и формулу обновления оставшихся весов для компенсации.
- Современные варианты: WoodFisher, M-FAC, OBC/SparseGPT — приближают обратный гессиан и применимы к моделям с миллиардами параметров.

### SparseGPT и Wanda (для LLM)

Прунинг больших языковых моделей оказался отдельной задачей: дообучение слишком дорогое, нужен one-shot подход.

- SparseGPT (Frantar & Alistarh, 2023) решает послойную задачу реконструкции активаций через приближённый OBS, обрабатывая колонки матрицы по очереди. Позволяет довести LLaMA до 50–60% разреженности почти без потерь.
- Wanda (Sun et al., 2023) ещё проще: важность веса оценивается как $|w_{ij}| \cdot \|x_j\|_2$, где $x_j$ — норма соответствующей входной активации на калибровочной выборке. Никакого решения уравнений, никакого дообучения, результаты сравнимы со SparseGPT.

### Lottery Ticket Hypothesis

Frankle & Carbin (2019) выдвинули гипотезу: в случайно инициализированной плотной сети существует разреженная подсеть («выигрышный билет»), которая, будучи обученной с того же начального состояния, достигает качества полной сети. Алгоритм поиска:

1. Инициализировать сеть весами $\theta_0$.
2. Обучить до сходимости, получить $\theta_T$.
3. Обнулить $p\%$ наименьших по модулю весов.
4. Сбросить оставшиеся веса к их значениям из $\theta_0$.
5. Повторять.

На больших сетях вместо $\theta_0$ используются веса с ранней эпохи обучения (rewinding). Гипотеза породила огромное количество исследований о роли инициализации и динамики обучения.

### Movement pruning

Sanh et al. (2020) заметили, что при дообучении предобученных моделей magnitude pruning работает хуже: важно не значение веса, а то, как он *движется* во время дообучения. Скоринг:

$$
S_{ij} = -\sum_t \frac{\partial \mathcal{L}}{\partial w_{ij}} \cdot w_{ij}
$$

Обнуляются веса, которые во время дообучения двигаются к нулю. Метод хорошо работает для transfer learning на BERT и подобных.

### Структурированные методы

- $\ell_1$/$\ell_2$-регуляризация на масштабах BatchNorm (Liu et al., *Network Slimming*) — каналы с малым $\gamma$ удаляются.
- Filter pruning по норме — удаление свёрточных фильтров с наименьшей $\ell_1$- или $\ell_2$-нормой.
- Taylor expansion (Molchanov et al.) — оценка изменения лосса при удалении канала через первый член разложения.
- LLM-Pruner, Sheared LLaMA — структурированный прунинг трансформеров (heads, MLP-нейроны, слои) с последующим коротким дообучением.

## Что важно при сравнении методов

При чтении статей по прунингу легко обмануться. На что стоит смотреть:

- Какой baseline сравнивается. Magnitude pruning часто оказывается неожиданно сильным; сложные методы порой выигрывают только за счёт настройки.
- Реальное ускорение vs теоретическое. 90% разреженности на CPU/GPU без sparse-ядер может вообще не дать прироста скорости.
- Бюджет на дообучение. Метод, требующий полного цикла обучения после прунинга, нечестно сравнивать с one-shot подходом.
- Сравнение по FLOPs vs по параметрам vs по реальной latency — разные метрики дают разные победители.
- Точка прунинга на кривой sparsity-accuracy. До 50–70% почти всё работает; интерес начинается на 90%+ для CNN и 50%+ для LLM.

## Аппаратные аспекты

- Плотные GPU без поддержки sparsity не дают ускорения от unstructured pruning, какой бы степени он ни был.
- N:M sparsity (2:4) на NVIDIA Ampere/Hopper даёт честный ~2x speedup на матричных умножениях.
- Структурированный прунинг (удаление каналов, голов, слоёв) ускоряет инференс на любом железе и совместим с экспортом в ONNX, TensorRT и т.д.
- На CPU и edge-устройствах структурированный прунинг почти всегда практичнее.

## Связь с другими техниками сжатия

Прунинг редко используется в изоляции. Типичные комбинации:

- Прунинг + квантизация — Deep Compression (Han et al.) показал, что они почти не интерферируют. Современные пайплайны для LLM (например, SparseGPT + GPTQ) идут именно так.
- Прунинг + дистилляция знаний — учитель «направляет» дообучение прунированной модели.
- Прунинг + low-rank decomposition — структурированное приближение матриц весов малого ранга, концептуально близкое.

## Практические рекомендации

- Для CNN и средних моделей magnitude pruning с iterative fine-tuning остаётся очень разумной отправной точкой.
- Если нужно реальное ускорение — сразу смотреть в сторону структурированного или 2:4-прунинга, а не unstructured.
- Для LLM без бюджета на дообучение — Wanda или SparseGPT, цель 50% разреженности.
- При высоких степенях разреженности (>90%) почти всегда нужен iterative подход, не one-shot.
- Перед прунингом стоит проверить, не решит ли задачу более дешёвая дистилляция в меньшую плотную модель — для многих практических случаев это эффективнее.

## Литература для углубления

- Han et al. *Learning both Weights and Connections for Efficient Neural Networks* (2015)
- Frankle & Carbin. *The Lottery Ticket Hypothesis* (ICLR 2019)
- Blalock et al. *What is the State of Neural Network Pruning?* (MLSys 2020) — критический обзор и стандартизация бенчмарков
- Hoefler et al. *Sparsity in Deep Learning* (JMLR 2021) — обширный survey
- Frantar & Alistarh. *SparseGPT* (ICML 2023)
- Sun et al. *A Simple and Effective Pruning Approach for Large Language Models* (Wanda, 2023)


__Distributed representation__ - гипотеза, которую поддерживали Хинтон, Румельхарт, Смоленски и другие. Базовый тезис: знание в нейросети не локализовано в отдельных весах или нейронах, оно размазано по их совокупности

Где на практике используется?<br>
- Мобильные и edge-устройства (память и энергопотребление критичны). Модные TPU поддерживают разреженные модели на уровне ядер. (модели для on-device speech recognition, computer vision на смартфонах, keyboard autocorrect)
- Тензорные ядра аппаратно умеют ускорять матричные умножения с 2:4 разреженностью примерно вдвое. Используется в инференсе LLM в крупных датацентрах, хотя и не повсеместно
- LLM inference сервисы. Здесь ситуация быстро меняется. SparseGPT, Wanda, SparseLLM применяются в продакшене при развёртывании открытых моделей на собственной инфраструктуре. NVIDIA TensorRT-LLM, Neural Magic (DeepSparse) — коммерческие решения, специализирующиеся именно на разреженном инференсе.
- Структурированный прунинг как часть NAS-пайплайнов. EfficientNet, MobileNet и подобные семейства фактически выросли из идеи структурированного прунинга, объединённого с поиском архитектуры

Почему не повсеместно используется<br>
- при прочих равных дистилляция и квантизация почти всегда проще и дают сравнимый результат
- реальный выигрыш в latency требует либо специализированного железа (sparse cores), либо специализированных рантаймов (DeepSparse на CPU). Это не везде доступно
- Для критичных production-моделей запас качества часто меньше, чем подсказывают академические бенчмарки на ImageNet или WikiText. Реальные распределения данных «в дикой природе» содержат хвосты, на которых прунинг бьёт сильнее, чем на тестовых сплитах

### Training-time pruning

Делать прунинг не постфактум, а с самого начала обучать модель так, чтобы она была разреженной или склонной к разреженности

$\ell_1$-регуляризация и Lasso. Самый старый и прямой подход. Добавляем к лоссу штраф $\lambda \|\theta\|_1$, и градиентный спуск естественно «давит» веса к нулю<br>обычно уступает iterative magnitude pruning.

$\ell_0$-регуляризация. Прямая формулировка задачи прунинга (минимизация числа ненулевых весов) недифференцируема, но её можно релаксировать. Louizos, Welling & Kingma (*Learning Sparse Neural Networks through L0 Regularization*, 2018) предложили использовать стохастические гейты на каждом весе с непрерывной релаксацией, дающей дифференцируемый surrogate для $\ell_0$. Идея красивая и работает, но в production-моделях прижилась слабо из-за сложности настройки.

Variational Dropout. Molchanov et al. (2017) показали, что байесовский dropout с обучаемой dispersion для каждого веса при правильной формулировке приводит к экстремальному разрежению — некоторые веса получают такую большую неопределённость, что фактически отключаются.

Sparse training (rigl, SET, top-K SGD). Целое семейство методов, поддерживающее разреженность *на протяжении всего обучения*, но позволяющее маске меняться.

- SET (Sparse Evolutionary Training, Mocanu et al., 2018). Стартуем со случайной разреженной маски. Каждые несколько эпох удаляем веса с наименьшим magnitude и случайно добавляем новые. Сеть «эволюционирует» к хорошей маске.
- RigL (Evci et al., 2020). То же, но новые связи добавляются не случайно, а по величине градиента — то есть выбираются те, которые сильнее всего хотели бы быть ненулевыми. Один из наиболее сильных методов sparse training: при той же разреженности часто догоняет качество прунинга плотной модели, при этом *ни на одном шаге обучения сеть не плотная*.
- Top-K SGD и подобные: на каждом шаге градиент применяется только к top-K весам по какому-то критерию.

Преимущество sparse training — обучение может быть в принципе дешевле, потому что и forward, и backward работают с разреженной сетью. На практике без специализированного железа этот выигрыш часто не реализуется, но методологически направление очень активное.

Lottery Ticket-вдохновлённые методы обучения. Если выигрышная подсеть существует с самого начала, можно попытаться найти её рано и обучать только её. Early-Bird tickets (You et al., 2020) показали, что маску можно надёжно определить уже после нескольких эпох обучения, после чего продолжить обучать только подсеть.

Архитектурная разреженность через проектирование. Mixture-of-Experts (MoE) — концептуально близкая идея на уровне архитектуры: для каждого входа активируется только небольшая часть параметров. Switch Transformer, Mixtral, GShard — production-системы. Это не прунинг в классическом смысле, но принцип тот же: «модель большая, но используется не вся сразу». MoE сейчас одно из самых горячих направлений в LLM именно потому, что даёт реальный выигрыш в инференсе на стандартном железе.

---

#### Показатели плотности модели

__Critical / breakdown sparsity__ с какого момента качество резко проседает

__Effective rank__ матриц весов. Для каждой матрицы $W$ можно посчитать сингулярные значения $\sigma_1, \ldots, \sigma_n$ и определить нормализованную энтропию спектра:

$$
\text{erank}(W) = \exp\left(-\sum_i p_i \log p_i\right), \quad p_i = \frac{\sigma_i}{\sum_j \sigma_j}
$$

Если эффективный ранг сильно меньше номинального — матрица содержит избыточные направления, потенциал для сжатия высок. Это используется в low-rank методах (LoRA и его родственники), но концептуально близко к прунингу.

__Intrinsic dimension__<br>
[Li et al. Measuring the Intrinsic Dimension of Objective Landscapes*, 2018](https://arxiv.org/abs/1804.08838)<br>
Какова минимальная размерность случайного подпространства в пространстве параметров, в котором всё ещё существует решение задачи с приемлемым качеством? Для MNIST это сотни-тысячи параметров, для ImageNet — десятки тысяч (при миллионах номинальных)

Li et al. на разных задачах нашли удивительно низкие d∗d^*d∗:
- MNIST с полносвязной сетью: d∗≈750d^* \approx 750 d∗≈750 (при D∼200,000D \sim 200{,}000 D∼200,000)
- CIFAR-10 со свёрточной сетью: d∗≈2,900d^* \approx 2{,}900 d∗≈2,900 (при D∼60,000,000D \sim 60{,}000{,}000 D∼60,000,000)
- ImageNet, более сложные задачи: d∗d^* d∗ десятки тысяч (при DD D в миллионах)

В работе Aghajanyan et al. (2020) [Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning](https://arxiv.org/abs/2012.13255) изучили потенциал сжатия трансофрмерных моделей<br>
d∗ для дообучения BERT и RoBERTa на GLUE-задачах — порядка сотен-тысяч параметров, при том что сама модель имеет сотни миллионов. Эта работа прямо мотивировала LoRA — раз эффективная размерность fine-tuning маленькая, дообучение можно ограничить low-rank поправками

__Compression ratio__<br>
во сколько раз модель можно сжать (pruning + quantization + distillation), потеряв не более X% качества. Используется в индустриальных бенчмарках типа MLPerf

---

#### Сравнение CV и NLP

Потенциал сокращения размерности в зависимости 
- Свёрточные сети: 90–95%
- Трансформеры (ViT): 70–85%
- LLM (GPT, BERT): 50–60%

Откуда возникает разница
- cвёртки в CV сильно перепараметризованы: один и тот же фильтр применяется ко всем пространственным позициям, что даёт огромное weight sharing на уровне *использования*, но при этом реальное разнообразие выученных фильтров часто мало. Трансформеры структурно более «острые» — каждый attention head учится своему распределению
- размерность выходного многообразия в CV мала (1000 классов у ImageNet). В NLP же мы моделируем в пространстве размерности словаря (десятки тысяч) и держим большой контекст => мы должны уложить больше информации и "знание" распределено по весам более равномерно
- в ImageNet мы обучаемся на ~1.3M картинок, ~25M параметров (ResNet-50). В LLM: триллионы токенов, миллиарды параметров => в CV прунинг работает как регуляризация и часто улучшает результат. Для LLM соотношение данные/параметры намного выше, и каждый параметр действительно используется на пределе своей информационной ёмкости

В 2022 году авторы исследования Chinchilla сказали, что неверно отдавать предпочтение размеру модели (как это делалось последние два года), а нужно размер модели увеличивать строго пропорционально продолжительности обучения. В итоге GPT модели стали обучать на большем кол-ве данных при меньшей размерности (повысив нагрузку на параметр). В 2023 году в работе Scaling Laws for Sparsity авторы уже напрямую изучали свойство сжимаемости

Итого, плотность модели — это свойство *совокупности* (архитектура, задача, объём данных, режим обучения)

---

#### Дистилляция

[(2015) Distilling the knowledge in a neural network](https://arxiv.org/abs/1503.02531)

Способ понижения размерности

### Intrinsic dimension

*Intrinsic dimension* (внутренняя размерность) задачи — это минимальная размерность подпространства в пространстве параметров, в котором всё ещё существует решение задачи с приемлемым качеством

Пусть модель имеет $D$ параметров (полное пространство $\theta \in \mathbb{R}^D$). Вместо обучения $\theta$ напрямую, фиксируем случайную точку $\theta_0 \in \mathbb{R}^D$ и случайную матрицу $P \in \mathbb{R}^{D \times d}$ со столбцами, образующими случайное $d$-мерное подпространство. Параметризуем модель через вектор $\eta \in \mathbb{R}^d$ ($d \ll D$):

$$
\theta = \theta_0 + P \eta
$$

Обучаем только $\eta$ — то есть оптимизация идёт в случайном $d$-мерном аффинном подпространстве $\mathbb{R}^D$.

*Intrinsic dimension* $d^*$ — минимальное $d$, при котором обучение в этом ограниченном подпространстве достигает заранее выбранного порога качества (например, 90% точности от full-параметрической модели).

#### История
During the 1950s so called "scaling" methods were developed in the social sciences to explore and summarize multidimensional data sets. After Shepard introduced non-metric multidimensional scaling in 1962 one of the major research areas within multi-dimensional scaling (MDS) was estimation of the intrinsic dimension. The topic was also studied in information theory, pioneered by Bennet in 1965 who coined the term "intrinsic dimension" and wrote a computer program to estimate it

During the 1970s intrinsic dimensionality estimation methods were constructed that did not depend on dimensionality reductions such as MDS: based on local eigenvalues., based on distance distributions, and based on other dimension-dependent geometric properties

Estimating intrinsic dimension of sets and probability measures has also been extensively studied since around 1980 in the field of dynamical systems, where dimensions of (strange) attractors have been the subject of interest. For strange attractors there is no manifold assumption, and the dimension measured is some version of fractal dimension — which also can be non-integer. However, definitions of fractal dimension yield the manifold dimension for manifolds.

In the 2000s the "curse of dimensionality" has been exploited to estimate intrinsic dimension

В отношении к полному размеру модели $d^*/D$ — обычно меньше 1%, иногда меньше 0.1%.

Аналогичный результат для fine-tuning Aghajanyan et al. (*Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning*, 2020): $d^*$ для дообучения BERT и RoBERTa на GLUE-задачах — порядка сотен-тысяч параметров, при том что сама модель имеет сотни миллионов. Эта работа прямо мотивировала LoRA — раз эффективная размерность fine-tuning маленькая, дообучение можно ограничить low-rank поправками.

Несколько важных свойств:
- $d^*$ зависит от задачи и порога качества, а не только от модели. Одна и та же сеть на разных датасетах даёт разный $d^*$.
- $d^*$ не зависит от выбора случайного подпространства (с точностью до концентрации меры): любая случайная $d^*$-мерная подматрица $P$ даёт примерно тот же результат, что фиксирует $d^*$ как свойство задачи, а не выбора параметризации.
- $d^* \leq D$ всегда; равенство означало бы, что задача использует всю ёмкость модели, чего на практике почти не бывает.

## Связь с прунингом

Концептуально близкие, но разные понятия. Intrinsic dimension говорит о минимальном *числе степеней свободы* для решения задачи в случайном базисе. Прунинг ищет минимальное число *исходных параметров*, способных решить задачу. Они оба меряют избыточность модели, но через разные геометрические объекты:

- Intrinsic dimension — про размерность многообразия решений.
- Прунинг — про разреженность представления решений в исходном базисе.

Эмпирически они коррелируют (модели с низким $d^*$ обычно сильно сжимаются прунингом), но не совпадают: intrinsic dimension может быть низким даже у плотной сети, где конкретно прунинг работает плохо.

## Интуиция

Можно представить так. Многообразие «приемлемых» решений в пространстве параметров — это какое-то подмножество $\mathbb{R}^D$. Если это многообразие имеет малую размерность, его легко «попасть» случайным $d$-мерным подпространством при достаточно большом $d$. Минимальное $d$, при котором случайное подпространство почти наверняка пересекает многообразие — это и есть intrinsic dimension.

Низкий $d^*$ означает «многообразие решений толстое» — попасть в него легко из любого направления. Высокий $d^*$ означает «многообразие тонкое» — нужно много направлений, чтобы гарантированно его задеть.

Это согласуется со всеми нашими предыдущими разговорами про многообразие эквивалентных решений, перепараметризацию и роль избыточности в нейросетях.